### Article Keyword Extraction (TF-IDF)

This module extracts the most important keywords from each news article using TF-IDF.

- **TF (Term Frequency):** Measures how often a word appears in an article.
- **IDF (Inverse Document Frequency):** Reduces the weight of common words across all articles.
- **Preprocessing:** Removes backslashes and extra whitespace from text.
- **Feature Extraction:** Uses TfidfVectorizer to convert text into numerical form.
- **Output:** Top 10 highest-scoring keywords are selected for each article.

**Purpose:**  
Identify meaningful words that best represent the content of each article.

In [4]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

ARTICLE_PATH = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\us_newsarticle_August_1"

def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    raw_articles = content.split('**************')

    articles_data = []
    for art in raw_articles:
        if art.strip():
            lines = art.strip().split('\n')

            title_match = re.search(r'-----(.*?)-----', lines[0])
            title = title_match.group(1).strip() if title_match else "Untitled"

            body = " ".join(lines[1:]).strip()

            articles_data.append({
                "Title": title,
                "Content": body
            })

    return pd.DataFrame(articles_data)

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df = load_dataset(ARTICLE_PATH)
df['Cleaned_Content'] = df['Content'].apply(preprocess_text)

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['Cleaned_Content'])
feature_names = vectorizer.get_feature_names_out()


for idx in range(len(df)):
    article_vector = tfidf_matrix[idx]

    df_tfidf = pd.DataFrame(
        article_vector.T.todense(),
        index=feature_names,
        columns=["score"]
    )

    top_10 = df_tfidf.sort_values(by="score", ascending=False).head(10)

    print(f"\nArticle {idx+1}: {df['Title'].iloc[idx]}")
    print("-" * 50)
    print(top_10)


Article 1: Melania Trump's girl-on-girl photos from racy shoot revealed
--------------------------------------------------
               score
melania     0.385404
photo       0.269783
basseville  0.238043
beauty      0.169216
eriksson    0.142826
featured    0.142826
shoot       0.142826
fashion     0.142826
source      0.126912
pictures    0.126912

Article 2: Ex-Venezuelan narcotics officials face drug-trafficking charges
--------------------------------------------------
                  score
brooklyn       0.330827
molina         0.248206
drug           0.200929
allegedly      0.200929
federal        0.153652
antidrug       0.124103
exgeneral      0.124103
antinarcotics  0.124103
deportation    0.124103
drugs          0.124103

Article 3: Props used in iconic movies to hit the auction block
--------------------------------------------------
             score
auction   0.306492
movie     0.272342
rake      0.153246
prop      0.153246
jaws      0.153246
store     0.153246
film 

### Article–Tweet Relevance Matching (TF-IDF Based)

This module matches news articles with relevant tweets using TF-IDF keyword scoring.

- **Article Processing:** Extracts titles and content from raw files and cleans text.
- **TF-IDF Extraction:** Converts articles into vectors and selects top 10 keywords per article.
- **Tweet Parsing:** Reads tweet files and separates tweet ID and text.
- **Relevance Scoring:** Compares tweet words with article keywords and sums their TF-IDF weights.
- **Ranking:** Tweets are sorted based on relevance score.

**Output:**  
Top 10 most relevant tweets are displayed for each article along with their scores.

**Purpose:**  
Identify and rank tweets that are most closely related to each news article.

In [9]:
import os
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

ARTICLE_PATH = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\us_newsarticle_August_1"
TWEETS_FOLDER = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\ALL_TWEETS"

def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    raw_articles = content.split('**************')

    articles_data = []
    for art in raw_articles:
        if art.strip():
            lines = art.strip().split('\n')

            title_match = re.search(r'-----(.*?)-----', lines[0])
            title = title_match.group(1).strip() if title_match else "Untitled"

            body = " ".join(lines[1:]).strip()

            articles_data.append({
                "Title": title,
                "Content": body
            })

    return pd.DataFrame(articles_data)

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = load_dataset(ARTICLE_PATH)
df['Cleaned_Content'] = df['Content'].apply(preprocess_text)

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['Cleaned_Content'])
feature_names = vectorizer.get_feature_names_out()

def process_tweet_file(file_path):
    rows = []
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            parts = line.split('*****', 1)
            if len(parts) != 2:
                continue

            tweet_id = parts[0].strip()
            tweet_text = parts[1].strip()

            rows.append((tweet_id, tweet_text))

    return rows

def calculate_relevance(text, keyword_weights):
    words = set(preprocess_text(text).split())
    score = 0

    for word, weight in keyword_weights.items():
        if word in words:
            score += weight

    return score

for idx in range(len(df)):

    article_vector = tfidf_matrix[idx]

    df_tfidf = pd.DataFrame(
        article_vector.T.todense(),
        index=feature_names,
        columns=["score"]
    )

    top_10 = df_tfidf.sort_values(by="score", ascending=False).head(10)
    keyword_weights = top_10['score'].to_dict()

    all_relevant_tweets = []

    for filename in os.listdir(TWEETS_FOLDER):
        file_path = os.path.join(TWEETS_FOLDER, filename)

        if os.path.isfile(file_path):
            tweets = process_tweet_file(file_path)

            for tweet_id, tweet_text in tweets:
                score = calculate_relevance(tweet_text, keyword_weights)

                if score > 0:
                    all_relevant_tweets.append((tweet_id, tweet_text, score))

  
    all_relevant_tweets.sort(key=lambda x: x[2], reverse=True)
    top_10_tweets = all_relevant_tweets[:10]

  
    print(f"\nARTICLE: -----{df['Title'].iloc[idx]}-----")
    print("=" * 30)

    for i, (tweet_id, tweet_text, score) in enumerate(top_10_tweets, 1):
        print(f"{i}. [{round(score,4)}] {tweet_id}*****{tweet_text}")


ARTICLE: -----Melania Trump's girl-on-girl photos from racy shoot revealed-----
1. [0.5818] 760226772977725440*****#fashion #celebs #d3320 #amandabynes candid #photo classy #sexy beauty #style #buzz
2. [0.5818] 760090755893334018*****#fashion #beauty #amandabynes 8x10 #photo picture hot #sexy candid 6 #buzz #celebs
3. [0.5818] 760226772977725440*****#fashion #celebs #d3320 #amandabynes candid #photo classy #sexy beauty #style #buzz
4. [0.5818] 760090755893334018*****#fashion #beauty #amandabynes 8x10 #photo picture hot #sexy candid 6 #buzz #celebs
5. [0.5818] 760226772977725440*****#fashion #celebs #d3320 #amandabynes candid #photo classy #sexy beauty #style #buzz
6. [0.5818] 760090755893334018*****#fashion #beauty #amandabynes 8x10 #photo picture hot #sexy candid 6 #buzz #celebs
7. [0.5818] 760226772977725440*****#fashion #celebs #d3320 #amandabynes candid #photo classy #sexy beauty #style #buzz
8. [0.5818] 760090755893334018*****#fashion #beauty #amandabynes 8x10 #photo picture hot 

### Article–Tweet Relevance Matching with Deduplication

This module links news articles with relevant tweets using TF-IDF keyword scoring.

- **Article Processing:** Extracts and cleans article text.
- **TF-IDF Keywords:** Top 10 keywords are selected per article.
- **Tweet Processing:** Reads and cleans tweet text from files.
- **Relevance Scoring:** Tweets are scored based on presence of weighted keywords.
- **Filtering:** Only tweets containing the main keyword are considered.
- **Deduplication:** Duplicate tweets are removed using `tweet_id`.
- **Ranking:** Tweets are sorted by relevance score.

**Output:**  
Top 10 unique and most relevant tweets per article.


In [10]:
import os
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

ARTICLE_PATH = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\us_newsarticle_August_1"
TWEETS_FOLDER = r"C:\Users\Admin\Downloads\Telegram Desktop\IRE data\IRE data\ALL_TWEETS"

def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    raw_articles = content.split('**************')

    articles_data = []
    for art in raw_articles:
        if art.strip():
            lines = art.strip().split('\n')

            title_match = re.search(r'-----(.*?)-----', lines[0])
            title = title_match.group(1).strip() if title_match else "Untitled"

            body = " ".join(lines[1:]).strip()

            articles_data.append({
                "Title": title,
                "Content": body
            })

    return pd.DataFrame(articles_data)


def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df = load_dataset(ARTICLE_PATH)
df['Cleaned_Content'] = df['Content'].apply(preprocess_text)

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['Cleaned_Content'])
feature_names = vectorizer.get_feature_names_out()


def process_tweet_file(file_path):
    rows = []
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            parts = line.split('*****', 1)
            if len(parts) != 2:
                continue

            tweet_id = parts[0].strip()
            tweet_text = parts[1].strip()

            rows.append((tweet_id, tweet_text))

    return rows


def calculate_relevance(text, keyword_weights):
    clean_text = preprocess_text(text)
    score = 0

    for word, weight in keyword_weights.items():
        if word in clean_text:
            score += weight

    return score

for idx in range(len(df)):

    article_vector = tfidf_matrix[idx]

    df_tfidf = pd.DataFrame(
        article_vector.T.todense(),
        index=feature_names,
        columns=["score"]
    )

    top_10 = df_tfidf.sort_values(by="score", ascending=False).head(10)
    keyword_weights = top_10['score'].to_dict()

    main_keyword = list(keyword_weights.keys())[0]

    all_relevant_tweets = []

    for filename in os.listdir(TWEETS_FOLDER):
        file_path = os.path.join(TWEETS_FOLDER, filename)

        if os.path.isfile(file_path):
            tweets = process_tweet_file(file_path)

            for tweet_id, tweet_text in tweets:
                clean_text = preprocess_text(tweet_text)
                score = calculate_relevance(tweet_text, keyword_weights)

                if score > 0 and main_keyword in clean_text:
                    all_relevant_tweets.append({
                        "tweet_id": str(tweet_id),
                        "tweet_text": tweet_text,
                        "relevance_score": score
                    })

    relevant_df = pd.DataFrame(all_relevant_tweets)

    if not relevant_df.empty:
        relevant_df = relevant_df.drop_duplicates(subset=['tweet_id'])
        relevant_df = relevant_df.sort_values(by='relevance_score', ascending=False)
        top_10_tweets = relevant_df.head(10)

        print(f"\nArticle {idx+1}: {df['Title'].iloc[idx]}")
        print("=" * 50)

        for i, row in enumerate(top_10_tweets.itertuples(index=False), 1):
            print(f"{i}. [{round(row.relevance_score,4)}] {row.tweet_id}*****{row.tweet_text}")
    else:
        print(f"\nArticle {idx+1}: {df['Title'].iloc[idx]}")
        print("No relevant tweets found.")


Article 1: Melania Trump's girl-on-girl photos from racy shoot revealed
1. [0.798] 759924380403064832*****staging melania's next nude photos shoot. #trumpdebateexcuses
2. [0.6552] 760183961704333312*****limbaugh: melania #trump's nude 'girl-on-girl' photos 'might wrap up the lgbt vote'. really? #valuevoters #gop
3. [0.6552] 759969557264027648*****#viral melania trump: racy nude photos surface amid donald's campaign...
4. [0.6552] 759945666500005889*****#followme #f2f #ff new york tabloid publishes nude photos of melania trump: the new york ... #followback #follow
5. [0.6552] 759974074525200384*****#viral melania trump: racy nude photos surface amid donald's campaign...
6. [0.6552] 759973973878636544*****#viral melania trump: racy nude photos surface amid donald's campaign...
7. [0.6552] 759973948717080576*****#viral melania trump: racy nude photos surface amid donald's campaign...
8. [0.6552] 759974481372717057*****#viral melania trump: racy nude photos surface amid donald's campaign.

### Issue: Over-Filtering of Relevant Tweets

The updated logic introduces a strict filtering condition using a single main keyword.

- **Problem:**  
  Tweets are only selected if their score is greater than 0 and contain the top TF-IDF keyword (`main_keyword`).

- **Why it fails:**  
  The top keyword is often too specific, and relevant tweets may not include that exact word even if they are contextually related.

- **Failure Case Example:**  
  - Article: *"Syrian child actor killed in Aleppo"*  
  - Main keyword: "abdou"
  - Tweet: *"Syrian child actor killed in missile strike in Aleppo"*  
  - Issue: Tweet is relevant but does **not contain `"abdou"`**, so it gets filtered out.

- **Impact:**  
  All tweets get filtered out, resulting in *"No relevant tweets found"*.

- **Fix:**  
  Relax the condition by:
  - Removing the `main_keyword` constraint, OR  
  - Matching against multiple top keywords instead of just one.

**Conclusion:**  
Over-restrictive filtering reduces recall; broader keyword matching improves results.